<a href="https://colab.research.google.com/github/IQRAZAM/research-papers/blob/main/BuidlingTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F



In [3]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [4]:
def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attn = torch.softmax(scores, dim=-1)
    return torch.matmul(attn, v), attn


In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()

        q = self.w_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        k = self.w_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        v = self.w_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        out, attn = scaled_dot_product_attention(q, k, v)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        return self.w_o(out)


In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, dim_ff):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.ReLU(),
            nn.Linear(dim_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.mha(x))
        x = self.norm2(x + self.ff(x))
        return x


In [7]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, dim_ff=256, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, dim_ff) for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = self.pos(x)
        for layer in self.layers:
            x = layer(x)
        return self.fc(x)


In [8]:
import random
import torch.optim as optim

vocab_size = 50
model = TinyTransformer(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def generate_batch(batch_size=32, seq_len=10):
    X = torch.randint(1, vocab_size, (batch_size, seq_len))
    Y = X.clone()
    return X, Y

for epoch in range(200):
    X, Y = generate_batch()
    out = model(X)
    loss = criterion(out.view(-1, vocab_size), Y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")



Epoch 0, Loss: 4.0910
Epoch 20, Loss: 1.3613
Epoch 40, Loss: 0.3844
Epoch 60, Loss: 0.1540
Epoch 80, Loss: 0.0888
Epoch 100, Loss: 0.0609
Epoch 120, Loss: 0.0455
Epoch 140, Loss: 0.0357
Epoch 160, Loss: 0.0284
Epoch 180, Loss: 0.0239


In [9]:
test = torch.randint(1, vocab_size, (1, 10))
pred = model(test).argmax(-1)
print("Input:", test)
print("Output:", pred)


Input: tensor([[22, 43, 36,  7, 36, 33, 21, 13, 22, 25]])
Output: tensor([[22, 43, 36,  7, 36, 33, 21, 13, 22, 25]])
